In [1]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook', 'grid'])

In [2]:
N = 250
t, m1, m2, k1, k2, b = sp.symbols('t m1 m2 k1 k2 b')
x1, x2 = sp.symbols('x1, x2', cls=sp.Function)
x1 = x1(t)
x2 = x2(t)

In [3]:
x1_d = sp.diff(x1, t)
x1_dd = sp.diff(x1_d, t)

x2_d = sp.diff(x2, t)
x2_dd = sp.diff(x2_d, t)

In [4]:
T1 = m1*x1_d**2/2
T2 = m2*x2_d**2/2
T = T1 + T2

U1 = k1*x1**2/2 + k2*(x1-x2)**2/2
U2 = k2*x2**2/2
U = U1 + U2

L = (T-U)*sp.exp(b*t)
L


(-k1*x1(t)**2/2 - k2*(x1(t) - x2(t))**2/2 - k2*x2(t)**2/2 + m1*Derivative(x1(t), t)**2/2 + m2*Derivative(x2(t), t)**2/2)*exp(b*t)

In [5]:
LE1 = sp.diff(L,x1) - sp.diff(sp.diff(L,x1_d),t).simplify()
LE2 = sp.diff(L,x2) - sp.diff(sp.diff(L,x2_d),t).simplify()
sols = sp.solve([LE1, LE2], (x1_dd, x2_dd), simplify = False, rational=False)
LE1

-m1*(b*Derivative(x1(t), t) + Derivative(x1(t), (t, 2)))*exp(b*t) + (-k1*x1(t) - k2*(2*x1(t) - 2*x2(t))/2)*exp(b*t)

In [6]:
dv1dt_f = sp.lambdify((t, m1, m2, k1, k2, b, x1, x2, x1_d, x2_d), sols[x1_dd])
dv2dt_f = sp.lambdify((t, m1, m2, k1, k2, b, x1, x2, x1_d, x2_d), sols[x2_dd])
dx1dt_f = sp.lambdify(x1_d, x1_d)
dx2dt_f = sp.lambdify(x2_d, x2_d)

In [7]:
def dSdt(S, t, m1, m2, k1, k2, b):
    x1, v1, x2, v2 = S
    return [
        dx1dt_f(v1),
        dv1dt_f(t, m1, m2, k1, k2, b, x1, x2, v1, v2),
        dx2dt_f(v2),
        dv2dt_f(t, m1, m2, k1, k2, b, x1, x2, v1, v2)
    ]

In [8]:
t = np.linspace(0, 40, N)
m1 = 1
m2 = 1
k1 = 1
k2 = 1
b = 0

ans = sc.integrate.odeint(dSdt, y0=[1, 0, -1, 0], t=t, args=(m1,m2,k1,k2,b))

In [9]:
x1 = ans.T[0]
x2 = ans.T[2]
plt.plot(t,x1)
plt.plot(t,x2)


In [10]:
L1 = 3
L2 = 3

x1 = x1 + L1
x2 = x2 + L2 + x1


def animate(i):
    ln1.set_data([0, x1[i], x2[i]], [0, 0, 0])

In [11]:
from matplotlib import pyplot as plt, animation

fig, ax = plt.subplots()
ax.set_facecolor('k')
ax.set(xlim=(0, 12), ylim=(-4, 4))
ln1, = plt.plot([], [], 'ro--', markersize=8)

ani = animation.FuncAnimation(fig, animate, frames = N-1, interval = 50)
#ani.save(filename="/Users/hasan/Python Animations/Coupled Harmonic Oscillator.gif", writer="pillow", fps=30)